# 01 — 无隐变量：直接最大似然
任务：仅给定二维样本 $y_i$，学习一个能生成相似样本的模型。真实数据生成机制只用于制造数据，不参与拟合。

主动定义 $p_\theta(y)=\mathcal N(y;\mu,\Sigma)$；未知量 $\theta=(\mu,\Sigma)$；观测量 $y_i$。没有隐变量，因此模型直接定义了观测密度，不需要边缘化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history, plot_observations

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 定义模型并学习参数
直接假设二维观测服从高斯分布，均值和协方差均未知。对于独立样本，目标是 $\max_{\mu,\Sigma}\sum_i\log p_{\mu,\Sigma}(y_i)$。这个问题有解析解，因此不需要梯度迭代。后面章节仍沿用同一个 MLE 准则。

对固定参数，单点密度为 $p_{\mu,\Sigma}(y_i)$；对固定观测，它是参数的似然。独立观测的联合密度相乘，取 log 后变成求和。解析 MLE 为 $\mu=\frac1N\sum_i y_i$、$\Sigma=\frac1N\sum_i(y_i-\mu)(y_i-\mu)^T$。下面逐项计算；np.outer 表示向量外积。

In [ ]:
plot_observations(y)

# 解析 MLE 的第一部分：样本均值，mu 为二维向量。
mu = np.zeros(2)
for obs in y:
    mu += obs
mu /= len(y)

# 解析 MLE 的第二部分：样本协方差，分母为 N 而非 N-1。
Sigma = np.zeros((2, 2))
for obs in y:
    error = obs - mu
    Sigma += np.outer(error, error)
Sigma /= len(y)

# 给定训练得到的参数，直接定义观测分布 p(y)。
model = multivariate_normal(mu, Sigma)
# 独立样本的联合对数似然是各样本对数密度之和。
log_likelihood = 0.0
for obs in y:
    log_likelihood += model.logpdf(obs)

print('mean:', mu)
print('covariance:', Sigma)
print('mean log likelihood:', log_likelihood / len(y))

# 从拟合得到的分布采样，检验生成效果。
samples = []
for i in range(len(y)):
    samples.append(model.rvs(random_state=rng))
compare(y, np.array(samples))

这里的最大似然解是样本均值和以 $N$ 为分母的协方差。注意：每个观测点的概率密度可计算，但它不是该点的概率。

改变协方差或均值，观察似然和生成结果。单高斯无法表达三个分离的峰，下一章只改变模型结构。